# 청각장애인을 위한 자가 발음 학습 AI — 오디오→혀위치 추정 파이프라인

Deaf_ai 프로젝트에서 설계한 5단계 학습 파이프라인(로컬 스크립트 `01_prepare_data.py` ~ `05_realtime_inference.py`)을 Google Colab에서 실행할 수 있도록 옮긴 노트북입니다.

**구성**
1. 라이브러리 설치
2. Google Drive 마운트 & 경로 설정
3. 1단계: TaL 데이터 전처리
4. 2단계: eigentongue 베이스라인 (Ridge 회귀)
5. 3단계: 1D CNN 모델 학습
6. 4단계: 화자독립 평가
7. 5단계: 실시간 추론 데모 (⚠️ 로컬 PC 전용, Colab에서는 마이크 접근 불가)

**실행 전 필요한 것**: TaL80(또는 TaL1) 데이터셋을 Google Drive에 올려두고, "경로 설정" 셀의 `TAL_ROOT`를 그 경로로 지정하세요.


In [1]:
# librosa, soundfile, joblib는 Colab 기본 이미지에 없을 수 있어 설치합니다.
# numpy, scipy, scikit-learn, torch는 Colab에 기본 설치되어 있습니다.
!pip install -q librosa soundfile joblib

## 0. Google Drive 마운트 & 경로 설정

원본 로컬 경로 `C:\AI_study\Deafness_AI\04_audio_to_tongue`에 대응하는 Drive 경로를 아래에서 지정합니다. 전처리 결과(`processed/`)와 모델 체크포인트가 모두 이 폴더 밑에 저장되므로, Colab 런타임이 종료되어도 Drive에 남습니다.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive 상에서 이 프로젝트가 저장될 경로 (필요하면 원하는 경로로 수정하세요)
BASE_DIR = "/content/drive/MyDrive/Deaf_AI/Deafness_AI_04_audio_to_tongue"
os.makedirs(BASE_DIR, exist_ok=True)

PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# TaL80 샘플 데이터 경로 (Drive에 업로드한 TaL_Corpus_Sample/samples/core 폴더)
# core 폴더 밑에 화자별 폴더(01fi, 06fe, ...)가 있고 그 안에 {번호}_aud.wav / .ult / .param 파일이 있습니다.
# 전체 TaL80/TaL1을 쓰는 경우에는 이 경로를 그 데이터의 화자별 폴더가 있는 위치로 바꾸면 됩니다.
TAL_ROOT = "/content/drive/MyDrive/Deaf_AI/TaL_Corpus_Sample/samples/core"
os.environ["TAL_ROOT"] = TAL_ROOT

print("BASE_DIR     :", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("TAL_ROOT     :", os.environ["TAL_ROOT"])

Mounted at /content/drive
BASE_DIR     : /content/drive/MyDrive/Deaf_AI/Deafness_AI_04_audio_to_tongue
PROCESSED_DIR: /content/drive/MyDrive/Deaf_AI/Deafness_AI_04_audio_to_tongue/processed
TAL_ROOT     : /content/drive/MyDrive/Deaf_AI/TaL_Corpus_Sample/samples/core


### (참고) TaL 데이터 준비 방법

원본 대화에서 정리된 옵션들입니다 (노트북 사양 문제로 클라우드 환경을 추천했던 내용):

- **전체 TaL80(약 498GB)**: 화자 81명 전체. 화자독립 검증까지 하려면 필요.
- **화자 일부만(예: 10~20명, 약 61~123GB)**: 파이프라인이 화자 수가 적어도 동작하도록 설계되어 있어, 코드 수정 없이 화자 수만 줄여도 됩니다.
- **samples 디렉토리(화자/세션당 샘플 2개)**: 코드가 제대로 도는지부터 빠르게 확인할 때.
- **TaL1(전문 성우 1명, 약 2시간 분량, 약 49GB)**: 화자독립 검증은 못 하지만 1차 검증엔 충분. 시간 분량은 짧지만 세션 6회분의 원본 초음파가 포함돼 있어 용량 자체는 작지 않음.

다운로드한 데이터를 Google Drive의 `TAL_ROOT` 경로(위 셀에서 지정)에 업로드한 뒤 아래 1단계부터 실행하세요.

## 1단계: 데이터 전처리 — `01_prepare_data.py`

TaL 원시 데이터(.wav, .ult, .param)를 읽어 오디오는 멜 스펙트로그램으로, 초음파는 다운샘플링 후 PCA(eigentongue)로 3차원까지 압축합니다. 두 시계열의 프레임레이트가 다르므로(오디오 100fps, 초음파 ~80fps) 선형보간으로 시간축을 정렬하고, 화자 단위로 train/val/test를 분리합니다(프레임 단위로 섞지 않음 — 화자독립 평가가 목적).

In [3]:
# -*- coding: utf-8 -*-
"""
04_audio_to_tongue / 01_prepare_data.py

목적
----
TaL Corpus(오디오 + 초음파 혀 영상 페어 데이터)를 읽어서:
1. 오디오 → 멜 스펙트로그램 특징 추출
2. 초음파 프레임 → PCA(eigentongue) 기반 저차원 조음 파라미터로 축소
3. 두 시계열의 시간축을 맞춤(정렬)
4. 화자 단위로 train/val/test를 나눠 저장

TaL 데이터 형식 (원 논문 기준)
------------------------------
- {utt}.wav : 48kHz 16bit 오디오
- {utt}.ult : 원시 초음파 데이터. 프레임당 64 scanline x 842 echo return
  (uint8), 즉 프레임 하나가 64*842 바이트.
- {utt}.param : 초음파 메타데이터(fps 등)를 담은 텍스트 파일(key=value 형식으로 가정)
- {utt}.txt : 읽은 문장 텍스트

주의: .ult / .param 정확한 바이너리·필드 스펙은 배포처의 UltraSuite 툴킷
(https://github.com/UltraSuite/ultrasuite-tools) 문서를 통해 반드시
재확인하세요. 아래 파서는 논문에 명시된 스펙(64 scanline x 842 echo)을
기준으로 작성한 합리적 추정이며, 실제 파일에서 프레임 수가 안 맞으면
NUM_SCANLINES / NUM_ECHOES 값을 조정해야 할 수 있습니다.
"""

import json
import os
import random
from pathlib import Path

import librosa
import numpy as np
from sklearn.decomposition import PCA

# ----------------------------------------------------------------------
# 설정값
# ----------------------------------------------------------------------
TAL_ROOT = os.environ.get("TAL_ROOT", os.path.join(BASE_DIR, "TaL80"))
OUTPUT_DIR = PROCESSED_DIR

NUM_SCANLINES = 64  # 초음파 스캔라인 수 (TaL 논문 기준)
NUM_ECHOES = 842  # 스캔라인당 echo return 수 (TaL 논문 기준)
ULTRASOUND_FPS_DEFAULT = 80.0  # .param에서 못 읽으면 쓰는 기본값 (논문 기준 ~80fps)

AUDIO_SR = 16000
N_MELS = 40
HOP_LENGTH = 160  # 16000/160 = 100fps, 오디오 특징 프레임레이트

TONGUE_PARAM_DIM = 3  # eigentongue 상위 몇 개 성분을 조음 파라미터로 쓸지
DOWNSAMPLE_ULT_SHAPE = (32, 84)  # PCA 전에 초음파 프레임을 이 크기로 축소(연산량 절감)

RANDOM_SEED = 42
VAL_SPEAKER_RATIO = 0.1
TEST_SPEAKER_RATIO = 0.1


# ----------------------------------------------------------------------
# 초음파 읽기
# ----------------------------------------------------------------------
def read_param_file(param_path):
    """key=value 또는 key:value 형식의 .param 파일을 최대한 유연하게 파싱한다."""
    params = {}
    if not os.path.exists(param_path):
        return params
    with open(param_path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            for sep in ("=", ":"):
                if sep in line:
                    k, v = line.split(sep, 1)
                    params[k.strip()] = v.strip()
                    break
    return params


def get_ultrasound_fps(param_path):
    params = read_param_file(param_path)
    for key in ("Fps", "fps", "FramesPerSec", "FrameRate"):
        if key in params:
            try:
                return float(params[key])
            except ValueError:
                pass
    return ULTRASOUND_FPS_DEFAULT


def read_ultrasound_raw(ult_path, num_scanlines=NUM_SCANLINES, num_echoes=NUM_ECHOES):
    """.ult 파일을 (T, num_scanlines, num_echoes) uint8 배열로 읽는다."""
    raw = np.fromfile(ult_path, dtype=np.uint8)
    frame_size = num_scanlines * num_echoes
    n_frames = raw.size // frame_size
    if n_frames == 0:
        raise ValueError(f"{ult_path}: 파일 크기가 예상 프레임 크기보다 작습니다. "
                          f"NUM_SCANLINES/NUM_ECHOES 설정을 확인하세요.")
    raw = raw[: n_frames * frame_size]
    frames = raw.reshape(n_frames, num_scanlines, num_echoes)
    return frames


def downsample_frames(frames, target_shape=DOWNSAMPLE_ULT_SHAPE):
    """PCA 연산량을 줄이기 위해 초음파 프레임을 간단히 블록 평균으로 축소한다."""
    t, h, w = frames.shape
    th, tw = target_shape
    h_bin, w_bin = h // th, w // tw
    if h_bin < 1 or w_bin < 1:
        return frames.astype(np.float32)
    trimmed = frames[:, : th * h_bin, : tw * w_bin]
    reshaped = trimmed.reshape(t, th, h_bin, tw, w_bin)
    return reshaped.mean(axis=(2, 4)).astype(np.float32)


# ----------------------------------------------------------------------
# 오디오 특징
# ----------------------------------------------------------------------
def extract_audio_features(wav_path, sr=AUDIO_SR, n_mels=N_MELS, hop_length=HOP_LENGTH):
    y, _ = librosa.load(wav_path, sr=sr)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=hop_length)
    log_mel = librosa.power_to_db(mel).T  # (T, n_mels)
    return log_mel


# ----------------------------------------------------------------------
# 시간축 정렬
# ----------------------------------------------------------------------
def align_time_axes(audio_feats, tongue_params, ultrasound_fps, audio_fps):
    """오디오 프레임 수에 맞춰 조음 파라미터를 선형보간으로 리샘플링한다."""
    n_audio = audio_feats.shape[0]
    n_ult = tongue_params.shape[0]
    if n_ult < 2:
        return None

    audio_t = np.arange(n_audio) / audio_fps
    ult_t = np.arange(n_ult) / ultrasound_fps

    aligned = np.zeros((n_audio, tongue_params.shape[1]), dtype=np.float32)
    for d in range(tongue_params.shape[1]):
        aligned[:, d] = np.interp(audio_t, ult_t, tongue_params[:, d])
    return aligned


# ----------------------------------------------------------------------
# 화자 목록 찾기 & split
# ----------------------------------------------------------------------
def find_utterances(tal_root):
    """TaL80 디렉토리 구조를 순회하며 (speaker_id, utt_base_path) 목록을 만든다.

    실제 TaL 배포 디렉토리 구조는 화자별 폴더로 되어 있을 가능성이 높으므로,
    이 함수는 '화자 폴더 하위에 {utt}.wav/.ult/.param/.txt가 있다'는 가정으로
    작성되었습니다. 실제 압축 해제 후 구조가 다르면 이 부분만 수정하면 됩니다.
    """
    utterances = []
    root = Path(tal_root)
    if not root.exists():
        return utterances
    for speaker_dir in sorted(root.iterdir()):
        if not speaker_dir.is_dir():
            continue
        speaker_id = speaker_dir.name
        for wav_path in speaker_dir.glob("*aud*.wav"):
            # 무음(sil)/속삭임(whi) 발화는 제외하고, 발성(aud) 발화만 사용
            base = wav_path.with_suffix("")
            ult_path = base.with_suffix(".ult")
            param_path = base.with_suffix(".param")
            if ult_path.exists():
                utterances.append((speaker_id, str(base)))
    return utterances


def split_speakers(speaker_ids, seed=RANDOM_SEED):
    speakers = sorted(set(speaker_ids))
    rng = random.Random(seed)
    rng.shuffle(speakers)
    n = len(speakers)
    n_val = max(1, int(n * VAL_SPEAKER_RATIO))
    n_test = max(1, int(n * TEST_SPEAKER_RATIO))
    test_speakers = set(speakers[:n_test])
    val_speakers = set(speakers[n_test:n_test + n_val])
    train_speakers = set(speakers[n_test + n_val:])
    return {"train": train_speakers, "val": val_speakers, "test": test_speakers}


# ----------------------------------------------------------------------
# 메인 파이프라인
# ----------------------------------------------------------------------
def fit_pca_on_sample(utterances, max_frames_for_pca=20000):
    """전체 발화 중 일부를 샘플링해 PCA(eigentongue)를 학습한다."""
    print("[1/3] PCA(eigentongue) 학습을 위한 초음파 프레임 샘플링 중...")
    sample_vectors = []
    rng = random.Random(RANDOM_SEED)
    shuffled = utterances[:]
    rng.shuffle(shuffled)

    for speaker_id, base in shuffled:
        if len(sample_vectors) >= max_frames_for_pca:
            break
        try:
            frames = read_ultrasound_raw(base + ".ult")
        except Exception as e:
            print(f"  [건너뜀] {base}: {e}")
            continue
        small = downsample_frames(frames)
        flat = small.reshape(small.shape[0], -1)
        sample_vectors.append(flat)

    if not sample_vectors:
        raise RuntimeError("PCA 학습용 초음파 프레임을 하나도 읽지 못했습니다. "
                            "TAL_ROOT 경로와 데이터 구조를 확인하세요.")

    all_vectors = np.concatenate(sample_vectors, axis=0)
    pca = PCA(n_components=TONGUE_PARAM_DIM, random_state=RANDOM_SEED)
    pca.fit(all_vectors)
    print(f"  PCA 학습 완료. 설명된 분산 비율: {pca.explained_variance_ratio_.sum():.1%}")
    return pca


def process_utterance(speaker_id, base, pca, split_name, out_dir):
    wav_path = base + ".wav"
    ult_path = base + ".ult"
    param_path = base + ".param"

    audio_feats = extract_audio_features(wav_path)
    frames = read_ultrasound_raw(ult_path)
    small = downsample_frames(frames)
    flat = small.reshape(small.shape[0], -1)
    tongue_params = pca.transform(flat)  # (T_ult, TONGUE_PARAM_DIM)

    ult_fps = get_ultrasound_fps(param_path)
    audio_fps = AUDIO_SR / HOP_LENGTH

    aligned_tongue = align_time_axes(audio_feats, tongue_params, ult_fps, audio_fps)
    if aligned_tongue is None:
        return False

    utt_name = os.path.basename(base)
    out_path = os.path.join(out_dir, split_name, f"{speaker_id}__{utt_name}.npz")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    np.savez(out_path, audio=audio_feats.astype(np.float32),
              tongue=aligned_tongue.astype(np.float32), speaker=speaker_id)
    return True


def main():
    utterances = find_utterances(TAL_ROOT)
    print(f"총 {len(utterances)}개 발화를 찾았습니다. (TAL_ROOT={TAL_ROOT})")
    if not utterances:
        print("[안내] 데이터가 없어 종료합니다. TAL_ROOT 환경변수를 TaL80 압축 해제 경로로 지정하세요.")
        print("      예: set TAL_ROOT=D:\\datasets\\TaL80 (Windows)")
        return

    pca = fit_pca_on_sample(utterances)

    speaker_ids = [s for s, _ in utterances]
    splits = split_speakers(speaker_ids)
    print(f"화자 분할 — train: {len(splits['train'])}, val: {len(splits['val'])}, test: {len(splits['test'])}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    counts = {"train": 0, "val": 0, "test": 0}
    print("[2/3] 발화별 특징 추출 및 정렬 중...")
    for speaker_id, base in utterances:
        split_name = next((s for s in ("train", "val", "test") if speaker_id in splits[s]), "train")
        try:
            ok = process_utterance(speaker_id, base, pca, split_name, OUTPUT_DIR)
            if ok:
                counts[split_name] += 1
        except Exception as e:
            print(f"  [실패] {base}: {e}")

    print(f"[3/3] 완료. 저장된 발화 수: {counts}")

    # PCA 모델과 split 정보를 함께 저장 (이후 단계에서 재사용)
    import joblib
    joblib.dump(pca, os.path.join(OUTPUT_DIR, "eigentongue_pca.joblib"))
    with open(os.path.join(OUTPUT_DIR, "splits.json"), "w") as f:
        json.dump({k: sorted(v) for k, v in splits.items()}, f, ensure_ascii=False, indent=2)
    print(f"결과 저장 위치: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


총 154개 발화를 찾았습니다. (TAL_ROOT=/content/drive/MyDrive/Deaf_AI/TaL_Corpus_Sample/samples/core)
[1/3] PCA(eigentongue) 학습을 위한 초음파 프레임 샘플링 중...
  PCA 학습 완료. 설명된 분산 비율: 29.0%
화자 분할 — train: 64, val: 8, test: 8
[2/3] 발화별 특징 추출 및 정렬 중...
[3/3] 완료. 저장된 발화 수: {'train': 123, 'val': 16, 'test': 15}
결과 저장 위치: /content/drive/MyDrive/Deaf_AI/Deafness_AI_04_audio_to_tongue/processed


## 2단계: eigentongue 베이스라인 — `02_baseline_eigentongue.py`

가장 단순한 베이스라인: 오디오 특징(멜 스펙트로그램) 프레임 하나로 같은 시점의 조음 파라미터를 Ridge 회귀로 예측합니다. "오디오만으로 혀 위치를 얼마나 맞힐 수 있는가"의 하한선을 빠르게 확인하는 용도입니다.

In [4]:
# -*- coding: utf-8 -*-
"""
04_audio_to_tongue / 02_baseline_eigentongue.py

목적
----
가장 단순한 베이스라인: 오디오 특징(멜 스펙트로그램) 프레임 하나로
같은 시점의 조음 파라미터(eigentongue 계수, 01단계에서 만든 라벨)를
선형회귀로 예측한다.

이게 왜 첫 실험이어야 하는가
--------------------------
복잡한 딥러닝 모델을 만들기 전에, "오디오만으로 혀 위치를 얼마나
맞힐 수 있는지"의 하한선(lower bound)을 빠르게 확인하기 위함이다.
이 베이스라인보다 딥러닝 모델이 못하면, 애초에 딥러닝 쪽 설계가
잘못된 것이다. TaL 논문이 인용한 Fabre 외(2015)의 eigentongue +
얕은 신경망 방식과 같은 수준의 접근이다.
"""

import glob
import json
import os

import joblib
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats import pearsonr


def load_split(split_name):
    """해당 split의 모든 npz 파일에서 (audio_frame, tongue_frame) 쌍을 모은다."""
    files = glob.glob(os.path.join(PROCESSED_DIR, split_name, "*.npz"))
    X, y, speakers = [], [], []
    for fpath in files:
        data = np.load(fpath, allow_pickle=True)
        audio = data["audio"]  # (T, N_MELS)
        tongue = data["tongue"]  # (T, TONGUE_PARAM_DIM)
        speaker = str(data["speaker"])
        n = min(len(audio), len(tongue))
        X.append(audio[:n])
        y.append(tongue[:n])
        speakers.extend([speaker] * n)
    if not X:
        return None, None, None
    return np.concatenate(X), np.concatenate(y), np.array(speakers)


def evaluate(model, X, y, name):
    pred = model.predict(X)
    print(f"\n=== {name} 성능 ===")
    for d in range(y.shape[1]):
        r, _ = pearsonr(y[:, d], pred[:, d])
        r2 = r2_score(y[:, d], pred[:, d])
        print(f"  조음 파라미터 {d+1}번 축: 상관계수 r={r:.3f}, R^2={r2:.3f}")

    # 참고용 baseline: "그냥 항상 평균값으로 예측"했을 때의 R^2 (항상 0.0)
    print(f"  참고: 아무 정보 없이 평균값만 예측하면 R^2=0.0 입니다.")
    print(f"  위 R^2가 0에 가깝다면, 오디오 정보가 이 축 예측에 거의 기여하지 못한다는 뜻입니다.")
    return pred


def main():
    print("데이터 로딩 중...")
    X_train, y_train, _ = load_split("train")
    X_val, y_val, _ = load_split("val")
    X_test, y_test, spk_test = load_split("test")

    if X_train is None:
        print("[안내] 학습 데이터가 없습니다. 먼저 01_prepare_data.py를 실행하세요.")
        return

    print(f"train 프레임 수: {len(X_train)}, val: {len(X_val) if X_val is not None else 0}, "
          f"test: {len(X_test) if X_test is not None else 0}")

    # 정규화 (화자마다 다른 오디오 특징 스케일 보정)
    mean, std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-6
    X_train_n = (X_train - mean) / std

    model = Ridge(alpha=1.0)
    model.fit(X_train_n, y_train)

    if X_val is not None:
        evaluate(model, (X_val - mean) / std, y_val, "검증셋(val, 학습에 없던 화자)")

    if X_test is not None:
        evaluate(model, (X_test - mean) / std, y_test, "테스트셋(test, 화자독립 최종 평가)")

    os.makedirs(PROCESSED_DIR, exist_ok=True)
    joblib.dump({"model": model, "mean": mean, "std": std},
                os.path.join(PROCESSED_DIR, "baseline_ridge.joblib"))
    print(f"\n베이스라인 모델 저장 완료: {os.path.join(PROCESSED_DIR, 'baseline_ridge.joblib')}")


if __name__ == "__main__":
    main()


데이터 로딩 중...
train 프레임 수: 78560, val: 10653, test: 9761

=== 검증셋(val, 학습에 없던 화자) 성능 ===
  조음 파라미터 1번 축: 상관계수 r=0.603, R^2=0.332
  조음 파라미터 2번 축: 상관계수 r=0.191, R^2=-0.093
  조음 파라미터 3번 축: 상관계수 r=0.060, R^2=-0.138
  참고: 아무 정보 없이 평균값만 예측하면 R^2=0.0 입니다.
  위 R^2가 0에 가깝다면, 오디오 정보가 이 축 예측에 거의 기여하지 못한다는 뜻입니다.

=== 테스트셋(test, 화자독립 최종 평가) 성능 ===
  조음 파라미터 1번 축: 상관계수 r=0.483, R^2=0.219
  조음 파라미터 2번 축: 상관계수 r=0.092, R^2=-0.078
  조음 파라미터 3번 축: 상관계수 r=0.205, R^2=0.026
  참고: 아무 정보 없이 평균값만 예측하면 R^2=0.0 입니다.
  위 R^2가 0에 가깝다면, 오디오 정보가 이 축 예측에 거의 기여하지 못한다는 뜻입니다.

베이스라인 모델 저장 완료: /content/drive/MyDrive/Deaf_AI/Deafness_AI_04_audio_to_tongue/processed/baseline_ridge.joblib


## 3단계: CNN 모델 학습 — `03_train_model.py`

02번 베이스라인(프레임 단위·문맥 없음)보다 나은 성능을 위해, 오디오 특징의 시간적 문맥(앞뒤 프레임 흐름)을 보는 경량 1D CNN 모델을 학습합니다. Colab의 무료 GPU(T4 등)를 쓰면 `DEVICE`가 자동으로 `cuda`로 선택됩니다.

In [5]:
# -*- coding: utf-8 -*-
"""
04_audio_to_tongue / 03_train_model.py

목적
----
02번 베이스라인(Ridge 회귀, 프레임 단위·문맥 없음)보다 나은 성능을 위해,
오디오 특징의 시간적 문맥(앞뒤 프레임 흐름)을 보는 경량 1D CNN 모델을
학습한다.

설계 근거
--------
- 발음은 순간적인 스냅샷이 아니라 연속적인 움직임이므로, 앞뒤 프레임을
  같이 보는 것이 이론적으로 더 유리하다. (02번 베이스라인은 프레임
  하나만 보고 예측하므로 이 정보를 활용하지 못한다.)
- 다만 처음부터 무거운 모델(Transformer 등)을 쓰지 않고, 작은 1D CNN
  스택으로 시작한다 — 데이터 규모가 크지 않을 수 있어 과적합 위험이
  있기 때문에, 03_semg에서도 강조했던 '작은 데이터로 버티는' 원칙을
  동일하게 적용한다.

실행 전 준비
-----------
pip install torch --index-url https://download.pytorch.org/whl/cpu
(GPU가 있다면 공식 홈페이지에서 CUDA 버전에 맞는 명령어로 설치)
"""

import glob
import json
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr

CHUNK_LEN = 200  # 한 번에 학습에 넣을 프레임 길이 (약 2초, hop=10ms 기준)
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------------------------------------------------
# 데이터셋
# ----------------------------------------------------------------------
class TongueDataset(Dataset):
    """발화를 CHUNK_LEN 길이로 잘라 (오디오, 조음파라미터) 쌍을 반환한다."""

    def __init__(self, split_name, chunk_len=CHUNK_LEN):
        files = glob.glob(os.path.join(PROCESSED_DIR, split_name, "*.npz"))
        self.chunks = []
        for fpath in files:
            data = np.load(fpath, allow_pickle=True)
            audio = data["audio"].astype(np.float32)
            tongue = data["tongue"].astype(np.float32)
            n = min(len(audio), len(tongue))
            audio, tongue = audio[:n], tongue[:n]
            for start in range(0, max(1, n - chunk_len + 1), chunk_len):
                a_chunk = audio[start:start + chunk_len]
                t_chunk = tongue[start:start + chunk_len]
                if len(a_chunk) < chunk_len // 2:
                    continue  # 너무 짧은 조각은 버림
                self.chunks.append((a_chunk, t_chunk))

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        audio, tongue = self.chunks[idx]
        return torch.from_numpy(audio), torch.from_numpy(tongue), len(audio)


def collate_fn(batch):
    """가변 길이 청크를 배치 내 최대 길이에 맞춰 0으로 패딩한다."""
    max_len = max(item[2] for item in batch)
    n_mels = batch[0][0].shape[1]
    n_tongue = batch[0][1].shape[1]
    audio_batch = torch.zeros(len(batch), max_len, n_mels)
    tongue_batch = torch.zeros(len(batch), max_len, n_tongue)
    mask = torch.zeros(len(batch), max_len)
    for i, (audio, tongue, length) in enumerate(batch):
        audio_batch[i, :length] = audio
        tongue_batch[i, :length] = tongue
        mask[i, :length] = 1.0
    return audio_batch, tongue_batch, mask


# ----------------------------------------------------------------------
# 모델
# ----------------------------------------------------------------------
class AudioToTongueCNN(nn.Module):
    """경량 1D CNN: (B, T, n_mels) -> (B, T, tongue_dim)"""

    def __init__(self, n_mels=40, tongue_dim=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_mels, hidden, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
        )
        self.head = nn.Linear(hidden, tongue_dim)

    def forward(self, x):
        # x: (B, T, n_mels) -> Conv1d는 (B, C, T)를 기대하므로 transpose
        x = x.transpose(1, 2)
        h = self.net(x)
        h = h.transpose(1, 2)  # (B, T, hidden)
        return self.head(h)


# ----------------------------------------------------------------------
# 학습 / 평가 루프
# ----------------------------------------------------------------------
def masked_mse(pred, target, mask):
    diff2 = (pred - target) ** 2
    diff2 = diff2 * mask.unsqueeze(-1)
    return diff2.sum() / (mask.sum() * pred.shape[-1] + 1e-8)


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_batches = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for audio, tongue, mask in loader:
            audio, tongue, mask = audio.to(DEVICE), tongue.to(DEVICE), mask.to(DEVICE)
            pred = model(audio)
            loss = masked_mse(pred, tongue, mask)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(1, n_batches)


def compute_correlation(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for audio, tongue, mask in loader:
            audio = audio.to(DEVICE)
            pred = model(audio).cpu().numpy()
            m = mask.numpy().astype(bool)
            for b in range(pred.shape[0]):
                preds.append(pred[b][m[b]])
                targets.append(tongue[b][m[b]].numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    correlations = []
    for d in range(preds.shape[1]):
        r, _ = pearsonr(targets[:, d], preds[:, d])
        correlations.append(r)
    return correlations


def main():
    print(f"device: {DEVICE}")
    train_ds = TongueDataset("train")
    val_ds = TongueDataset("val")
    if len(train_ds) == 0:
        print("[안내] 학습 데이터가 없습니다. 먼저 01_prepare_data.py를 실행하세요.")
        return

    print(f"train 청크 수: {len(train_ds)}, val 청크 수: {len(val_ds)}")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    n_mels = train_ds.chunks[0][0].shape[1]
    tongue_dim = train_ds.chunks[0][1].shape[1]
    model = AudioToTongueCNN(n_mels=n_mels, tongue_dim=tongue_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_loss = float("inf")
    best_path = os.path.join(PROCESSED_DIR, "cnn_model_best.pt")

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader, optimizer=None)
        print(f"[epoch {epoch:02d}] train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({"model_state": model.state_dict(),
                        "n_mels": n_mels, "tongue_dim": tongue_dim}, best_path)

    if len(val_ds) > 0:
        corrs = compute_correlation(model, val_loader)
        print("\n=== 검증셋 화자독립 상관계수 (최종 epoch 기준) ===")
        for d, r in enumerate(corrs):
            print(f"  조음 파라미터 {d+1}번 축: r={r:.3f}")

    print(f"\n최적 모델 저장 위치: {best_path}")


if __name__ == "__main__":
    main()


device: cuda
train 청크 수: 327, val 청크 수: 46
[epoch 01] train_loss=103896.9107 val_loss=87181.0625
[epoch 02] train_loss=102981.0781 val_loss=87345.9349
[epoch 03] train_loss=103576.4275 val_loss=86423.5990
[epoch 04] train_loss=102908.4368 val_loss=86860.9688
[epoch 05] train_loss=102202.0406 val_loss=86450.7396
[epoch 06] train_loss=101693.4025 val_loss=87722.9062
[epoch 07] train_loss=101660.0543 val_loss=86016.4740
[epoch 08] train_loss=101450.0413 val_loss=88223.2943
[epoch 09] train_loss=101979.2999 val_loss=82510.3594
[epoch 10] train_loss=99562.1577 val_loss=85927.7891
[epoch 11] train_loss=99217.0117 val_loss=84210.5755
[epoch 12] train_loss=100786.1291 val_loss=84340.7396
[epoch 13] train_loss=100233.8553 val_loss=85060.7292
[epoch 14] train_loss=98628.9089 val_loss=86295.5781
[epoch 15] train_loss=97950.6246 val_loss=84211.5703
[epoch 16] train_loss=98738.8765 val_loss=80870.7656
[epoch 17] train_loss=96524.2820 val_loss=84032.3958
[epoch 18] train_loss=97048.4546 val_loss=808

## 4단계: 화자독립 평가 — `04_evaluate.py`

02번(Ridge 베이스라인)과 03번(CNN 모델)을 같은 테스트셋(학습에 전혀 없던 화자)에서 비교 평가합니다. "복잡한 모델을 만드는 게 실제로 의미가 있었는가"를 확인하는 단계입니다.

In [6]:
# -*- coding: utf-8 -*-
"""
04_audio_to_tongue / 04_evaluate.py

목적
----
02번(Ridge 베이스라인)과 03번(CNN 모델)을 같은 테스트셋(학습에 전혀
없던 화자)에서 비교 평가한다. "복잡한 모델을 만드는 게 실제로 의미가
있었는가"를 확인하는 단계다.
"""

import glob
import os

import joblib
import numpy as np
import torch
from scipy.stats import pearsonr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_cnn_model():
    ckpt_path = os.path.join(PROCESSED_DIR, "cnn_model_best.pt")
    if not os.path.exists(ckpt_path):
        return None
    # 03_train_model.py와 동일한 구조를 그대로 정의 (파일명이 숫자로 시작해 import가 번거로우므로 재정의)
    import torch.nn as nn

    class AudioToTongueCNN(nn.Module):
        def __init__(self, n_mels, tongue_dim, hidden=64):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(n_mels, hidden, kernel_size=5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
                nn.Conv1d(hidden, hidden, kernel_size=5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1), nn.BatchNorm1d(hidden), nn.ReLU(),
            )
            self.head = nn.Linear(hidden, tongue_dim)

        def forward(self, x):
            x = x.transpose(1, 2)
            h = self.net(x)
            h = h.transpose(1, 2)
            return self.head(h)

    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = AudioToTongueCNN(ckpt["n_mels"], ckpt["tongue_dim"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model


def load_test_utterances():
    files = glob.glob(os.path.join(PROCESSED_DIR, "test", "*.npz"))
    utterances = []
    for fpath in files:
        data = np.load(fpath, allow_pickle=True)
        utterances.append({
            "audio": data["audio"].astype(np.float32),
            "tongue": data["tongue"].astype(np.float32),
            "speaker": str(data["speaker"]),
            "name": os.path.basename(fpath),
        })
    return utterances


def evaluate_ridge(utterances):
    baseline_path = os.path.join(PROCESSED_DIR, "baseline_ridge.joblib")
    if not os.path.exists(baseline_path):
        print("[안내] 02_baseline_eigentongue.py를 먼저 실행하세요.")
        return None
    ckpt = joblib.load(baseline_path)
    model, mean, std = ckpt["model"], ckpt["mean"], ckpt["std"]

    preds, targets = [], []
    for utt in utterances:
        n = min(len(utt["audio"]), len(utt["tongue"]))
        x = (utt["audio"][:n] - mean) / std
        pred = model.predict(x)
        preds.append(pred)
        targets.append(utt["tongue"][:n])
    return np.concatenate(preds), np.concatenate(targets)


def evaluate_cnn(model, utterances):
    if model is None:
        print("[안내] 03_train_model.py를 먼저 실행하세요.")
        return None
    preds, targets = [], []
    with torch.no_grad():
        for utt in utterances:
            n = min(len(utt["audio"]), len(utt["tongue"]))
            x = torch.from_numpy(utt["audio"][:n]).unsqueeze(0).to(DEVICE)
            pred = model(x).cpu().numpy()[0]
            preds.append(pred)
            targets.append(utt["tongue"][:n])
    return np.concatenate(preds), np.concatenate(targets)


def report(name, preds, targets):
    if preds is None:
        return
    print(f"\n=== {name} — 화자독립 테스트셋 최종 성능 ===")
    for d in range(targets.shape[1]):
        r, _ = pearsonr(targets[:, d], preds[:, d])
        rmse = float(np.sqrt(np.mean((targets[:, d] - preds[:, d]) ** 2)))
        print(f"  조음 파라미터 {d+1}번 축: r={r:.3f}, RMSE={rmse:.3f}")


def main():
    utterances = load_test_utterances()
    if not utterances:
        print("[안내] 테스트 데이터가 없습니다. 01_prepare_data.py를 먼저 실행하세요.")
        return
    print(f"테스트 발화 수: {len(utterances)} (화자: {sorted(set(u['speaker'] for u in utterances))})")

    ridge_result = evaluate_ridge(utterances)
    report("02. Ridge 베이스라인 (문맥 없음)", *ridge_result if ridge_result else (None, None))

    cnn_model = load_cnn_model()
    cnn_result = evaluate_cnn(cnn_model, utterances)
    report("03. CNN 모델 (문맥 포함)", *cnn_result if cnn_result else (None, None))

    print("\n=== 해석 가이드 ===")
    print("- CNN 모델의 r이 Ridge보다 뚜렷이 높다면: 시간적 문맥이 실제로 도움이 된다는 뜻")
    print("- 두 모델 다 r이 0에 가깝다면: 오디오만으로 혀 위치 추정 자체가 이 데이터에서 어렵다는 뜻")
    print("  → 이 경우 화면 피드백의 신뢰도 표시(점선/흐림)를 더 강하게 적용해야 함")


if __name__ == "__main__":
    main()


테스트 발화 수: 15 (화자: ['09fe', '24fe', '31ms', '48ms', '51fe', '52fs', '57ms', '65ms'])

=== 02. Ridge 베이스라인 (문맥 없음) — 화자독립 테스트셋 최종 성능 ===
  조음 파라미터 1번 축: r=0.483, RMSE=416.778
  조음 파라미터 2번 축: r=0.092, RMSE=315.980
  조음 파라미터 3번 축: r=0.205, RMSE=150.072

=== 03. CNN 모델 (문맥 포함) — 화자독립 테스트셋 최종 성능 ===
  조음 파라미터 1번 축: r=0.568, RMSE=438.470
  조음 파라미터 2번 축: r=0.193, RMSE=301.151
  조음 파라미터 3번 축: r=0.359, RMSE=148.927

=== 해석 가이드 ===
- CNN 모델의 r이 Ridge보다 뚜렷이 높다면: 시간적 문맥이 실제로 도움이 된다는 뜻
- 두 모델 다 r이 0에 가깝다면: 오디오만으로 혀 위치 추정 자체가 이 데이터에서 어렵다는 뜻
  → 이 경우 화면 피드백의 신뢰도 표시(점선/흐림)를 더 강하게 적용해야 함


## 5단계: 실시간 추론 데모 — `05_realtime_inference.py`

⚠️ **Colab 제한사항**: 이 스크립트는 로컬 마이크(`sounddevice`)를 사용합니다. Colab 런타임은 클라우드 서버라 사용자의 마이크에 직접 접근할 수 없으므로, 이 셀은 **Colab에서 실행되지 않도록 주석 처리**해뒀습니다. 학습된 모델(`cnn_model_best.pt`)을 Drive에서 로컬 PC로 내려받은 뒤, 로컬 Python 환경에서 `python 05_realtime_inference.py`로 실행하세요.

In [ ]:
# -*- coding: utf-8 -*-
"""
04_audio_to_tongue / 05_realtime_inference.py

목적
----
학습된 CNN 모델(03단계 산출물)을 이용해, 실시간 마이크 입력으로부터
혀 위치 파라미터를 추정하고 콘솔에 간단히 시각화하는 데모.

이 스크립트는 화면 UI(웹캠 입모양 비교 + 혀 위치 참조 단면도)를
그대로 구현한 것은 아니고, "모델이 실시간으로 잘 도는지"를 빠르게
확인하기 위한 콘솔 버전입니다. 실제 앱에서는 이 추론 결과를
01_webcam_vsr의 입모양 결과와 함께 화면에 그리면 됩니다.
"""

import os
import time

import numpy as np
import sounddevice as sd
import torch
import torch.nn as nn
import librosa

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

AUDIO_SR = 16000
N_MELS = 40
HOP_LENGTH = 160
WINDOW_SECONDS = 1.0  # 추론 1회에 사용할 오디오 길이
UPDATE_INTERVAL = 0.5  # 추론 주기(초)

AXIS_NAMES = ["혀 전후 위치", "혀 고저 위치", "3번 축(해석 필요)"]


class AudioToTongueCNN(nn.Module):
    def __init__(self, n_mels, tongue_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_mels, hidden, kernel_size=5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1), nn.BatchNorm1d(hidden), nn.ReLU(),
        )
        self.head = nn.Linear(hidden, tongue_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        h = self.net(x)
        h = h.transpose(1, 2)
        return self.head(h)


def load_model():
    ckpt_path = os.path.join(PROCESSED_DIR, "cnn_model_best.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError("cnn_model_best.pt가 없습니다. 03_train_model.py를 먼저 실행하세요.")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = AudioToTongueCNN(ckpt["n_mels"], ckpt["tongue_dim"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model


def extract_features(audio):
    mel = librosa.feature.melspectrogram(y=audio, sr=AUDIO_SR, n_mels=N_MELS, hop_length=HOP_LENGTH)
    log_mel = librosa.power_to_db(mel).T
    return log_mel.astype(np.float32)


def render_bar(value, axis_range=(-50, 50), width=20):
    """-축범위~+축범위 값을 [-------|----] 형태의 텍스트 막대로 표시한다."""
    lo, hi = axis_range
    ratio = (value - lo) / (hi - lo)
    ratio = max(0.0, min(1.0, ratio))
    pos = int(ratio * width)
    bar = ["-"] * width
    bar[pos] = "●"
    return "[" + "".join(bar) + f"] {value:+.1f}"


def main():
    print("=" * 60)
    print(" Deafness_AI - 04. 오디오 기반 혀 위치 실시간 추론 데모")
    print("=" * 60)
    model = load_model()

    buffer = np.zeros(int(AUDIO_SR * WINDOW_SECONDS), dtype=np.float32)

    def callback(indata, frames, time_info, status):
        nonlocal buffer
        mono = indata[:, 0]
        buffer = np.roll(buffer, -len(mono))
        buffer[-len(mono):] = mono

    print("마이크 스트리밍을 시작합니다. Ctrl+C로 종료하세요.\n")
    with sd.InputStream(samplerate=AUDIO_SR, channels=1, callback=callback):
        try:
            while True:
                time.sleep(UPDATE_INTERVAL)
                feats = extract_features(buffer.copy())
                x = torch.from_numpy(feats).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    pred = model(x).cpu().numpy()[0]
                # 최근 프레임(윈도우 끝부분) 예측값을 대표값으로 사용
                latest = pred[-5:].mean(axis=0)

                print("\033c", end="")  # 콘솔 화면 지우기 (터미널에 따라 동작 안 할 수 있음)
                print("=== 실시간 혀 위치 추정 (참고용, 개략적 수치) ===")
                for i, name in enumerate(AXIS_NAMES[:len(latest)]):
                    print(f"{name:12s} {render_bar(latest[i])}")
                print("\n(Ctrl+C로 종료)")
        except KeyboardInterrupt:
            print("\n종료합니다.")


# Colab 콘솔에는 마이크가 없어 자동 실행하지 않습니다.
# 로컬 PC에 코드를 내려받은 뒤 아래 두 줄의 주석을 해제하고 실행하세요.
# if __name__ == "__main__":
#     main()
